In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
adj_close= pd.read_csv(
    "../data/adj_close.csv",
    index_col=0,
    parse_dates=True
)

adj_close.head()

# 1. Return calculation

In [ ]:
returns= (adj_close/adj_close.shift(1))-1
returns

# dropna() removes rows containing missing values (NaN).
returns = returns.dropna() saves the cleaned DataFrame back into returns. Here, it removes only the first row because returns cannot be calculated for the first trading day.

In [ ]:
returns = returns.dropna()
returns

# 2. PORTFOLIO WEIGHTS

# Since our portfolio contains 9 stocks, we assign an equal weight to each stock.
 weights = np.repeat(1/9, 9) i.e. np.repeat(value, repeats)

In [ ]:
weights= np.repeat(1/len(returns.columns), len(returns.columns)) #np.repeat(value, number_of_times)

weights

# 3. Portfolio Return = Σ (Weight × Asset Return)

In [ ]:
p_returns = np.dot(returns, weights) # a multiply + add operation.
p_returns

In [ ]:
len(p_returns)

# 4. Portfolio stats (daily return and volatility)

In [ ]:
p_returns.mean()

In [ ]:
p_returns.std()

In [ ]:
print(f"Average daily return is {p_returns.mean():.4%}")

In [ ]:
print(f'Risk of the portfolio is {p_returns.std(): .4%}')

# 5. ANNUAL RETURNS AND VOLATILITY

In [ ]:
annual_returns= p_returns.mean()* 252 
annual_returns

In [ ]:
#Uses √252 because risk (volatility) grows with the square root of time.

In [ ]:
#Annual Volatility = Daily Volatility × √252
annual_volatility= p_returns.std()* np.sqrt(252) 
annual_volatility

In [ ]:
print(f"Annual portfolio return: {annual_returns:.2%}")

In [ ]:
print(f"Annual portfolio volatility: {annual_volatility:.2%}")

# 6. SHARPE RATIO


Measures the return earned for each unit of risk.

Formula:

Sharpe Ratio = (Annual Return − Risk-Free Rate) / Annual Volatility

Higher Sharpe Ratio = Better risk-adjusted performance.

**Risk-free rate.** Because the investor is euro-based, we use €STR (euro short-term rate), the ECB's overnight rate and the official euro risk-free benchmark.

We use a **constant full-period approximation of 1.05% per year**: the simple average of the annual €STR averages for 2019–2025 (−0.54%, −0.55%, −0.57%, −0.01%, 3.21%, 3.64%, 2.18%). €STR started on 2 October 2019, so the 2019 figure covers only October–December, but it counts as a full year in the average.

For comparison, the ECB's earlier "pre-€STR" series covers January–September 2019. Estimating it as EONIA minus 8.5 basis points (the spread the ECB published between the two rates) gives a full-year 2019 average of about −0.48% and a seven-year average of about **1.06%**. The difference is 0.01 percentage points and changes the Sharpe ratio by less than 0.001, so the simpler 1.05% is kept.

Subtracting 1.05% from the annual return is the same as subtracting 1.05% ÷ 252 from every daily return, so no separate daily rate is needed.

In [ ]:
# €STR constant full-period approximation (average of annual €STR averages, 2019-2025).
# 2019 covers October-December only; a full-year pre-€STR estimate gives about 1.06%.
risk_free_rate = 0.0105

In [ ]:
sharpe_ratio= (annual_returns-risk_free_rate)/ annual_volatility
print(f'sharpe ratio is {sharpe_ratio}')

# 7. Cumulative returns

In [ ]:
cumulative_returns = (1 + p_returns).cumprod()

cumulative_returns

# 8. Maximum Drawdown 📉

In [ ]:
#running_peak
running_peak = np.maximum.accumulate(cumulative_returns)
#cummax()-It keeps track of the highest value reached so far.


In [ ]:
running_peak[:10] #This will show us the first 10 running-peak values.

In [ ]:
##Calculate Drawdown
drawdown = (cumulative_returns - running_peak) / running_peak
drawdown


In [ ]:
#max drawdown
max_drawdown = drawdown.min() #.min() Because drawdowns are negative:
print(f"Maximum Drawdown: {max_drawdown:.2%}")

In [ ]:
# Convert portfolio returns to a Pandas Series (p_returns is currently a NumPy array, which doesn't have dates or pandas' time-series functions.)
portfolio_returns = pd.Series(p_returns, index=returns.index)
portfolio_returns.head()

# 9. Rolling volatility

In [ ]:
# 30-day rolling volatility
rolling_volatility = portfolio_returns.rolling(window=30).std()
rolling_volatility

In [ ]:
# Annualize the volatility
rolling_volatility_annualized = rolling_volatility * np.sqrt(252)
rolling_volatility_annualized

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_volatility_annualized)
plt.title("30-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.grid(True)
plt.show()

# 10. Best and worst trading days

In [ ]:

best_day = portfolio_returns.max()
worst_day = portfolio_returns.min()

print(f"Best Day: {best_day:.2%}")
print(f"Worst Day: {worst_day:.2%}")

# 11. Positive and negative days

In [ ]:

positive_days = (portfolio_returns > 0).sum()
negative_days = (portfolio_returns < 0).sum()

total_days = len(portfolio_returns)

print(f"Positive Days: {positive_days} ({positive_days / total_days:.2%})")
print(f"Negative Days: {negative_days} ({negative_days / total_days:.2%})")

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(portfolio_returns, bins=50)
plt.title("Distribution of Daily Portfolio Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency") #how many trading days fell into each return range.
plt.grid(True)
plt.show()